# DuckDB Demo — GHArchive on Ozone

Queries GitHub Archive data from Apache Ozone using DuckDB's S3 connector.
DuckDB reads JSON and Parquet files directly from `s3://` — no Spark needed.

In [ ]:
import os
import duckdb

print("DuckDB version:", duckdb.__version__)

con = duckdb.connect()

ENDPOINT = os.environ["AWS_S3_ENDPOINT"].replace("http://", "").replace("https://", "")

# Configure S3 to point at Ozone using the Secrets API (DuckDB 1.x)
con.execute(f"""
    CREATE OR REPLACE SECRET ozone (
        TYPE S3,
        KEY_ID '{os.environ["AWS_ACCESS_KEY_ID"]}',
        SECRET '{os.environ["AWS_SECRET_ACCESS_KEY"]}',
        ENDPOINT '{ENDPOINT}',
        URL_STYLE 'path',
        USE_SSL false
    )
""")

print("DuckDB configured for Ozone at:", ENDPOINT)

## 1. Count events in a partition

In [ ]:
import s3fs, os
from botocore.config import Config

fs = s3fs.S3FileSystem(
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    key=os.environ["AWS_ACCESS_KEY_ID"],
    secret=os.environ["AWS_SECRET_ACCESS_KEY"],
    client_kwargs={"config": Config(s3={"addressing_style": "path"})},
)

files = fs.glob("raw/gh_archive/**/*.json.gz")
if not files:
    print("No raw data found. Run the download DAG in Airflow first.")
else:
    sample = "s3://" + files[0]
    print("Querying:", sample)

    result = con.execute(f"""
        SELECT COUNT(*) AS total_events
        FROM read_json_auto('{sample}', maximum_object_size=104857600)
    """).df()
    print(result)

## 2. Top event types with SQL

In [ ]:
if files:
    top = con.execute(f"""
        SELECT type, COUNT(*) AS cnt
        FROM read_json_auto('{sample}', maximum_object_size=104857600)
        GROUP BY type
        ORDER BY cnt DESC
        LIMIT 10
    """).df()
    print(top)

## 3. Query curated Iceberg Parquet files

In [ ]:
parquet_files = fs.glob("curated/curated/github_events/data/**/*.parquet")

if not parquet_files:
    print("No curated data yet — run iceberg_demo.ipynb first.")
else:
    paths = "', '".join("s3://" + f for f in parquet_files[:10])

    result = con.execute(f"""
        SELECT type, actor, COUNT(*) AS events
        FROM read_parquet(['{paths}'])
        GROUP BY type, actor
        ORDER BY events DESC
        LIMIT 20
    """).df()
    print(result)

## 4. Save a DuckDB query result to Ozone

In [ ]:
if files:
    con.execute(f"""
        COPY (
            SELECT type, COUNT(*) AS cnt
            FROM read_json_auto('{sample}', maximum_object_size=104857600)
            GROUP BY type
            ORDER BY cnt DESC
        )
        TO 's3://curated/duckdb_output/event_counts.parquet'
        (FORMAT parquet)
    """)
    print("Written to s3://curated/duckdb_output/event_counts.parquet")